# Ejercicio 5: Espacio Vectorial

## Objetivo de la práctica
- Implementar un Sistema de Recuperación de Información completo, desde la lectura del corpus hasta la recuperación de resultados.

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus
2. Realiza las etapas de preprocesamiento sobre el corpus


#### 1. Descarga del corpus desde kaggle

In [ ]:
import kagglehub

path = kagglehub.dataset_download("gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects")

print("Path to dataset files:", path)

c:\Users\david\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 18.7M/18.7M [00:06<00:00, 2.82MB/s]

Extracting files...


Path to dataset files: C:\Users\david\.cache\kagglehub\datasets\gzdekzlkaya\wikipedia-text-corpus-for-nlp-and-llm-projects\versions\1


#### 2. Importación de librerías y definición de librerías

Para desarrollar el ejercicio se utilizaron las siguientes librerías:
- *re:* Limpieza de caracteres especiales.
- *pandas:* Manipulación de data frames.
- *sklearn:* Implementación de TF-IDF y similitud del coseno.
- *numpy:* Manipulación de matrices.

Además se definió la variable path que se utilizará en toda la práctica

In [ ]:
import re
import pandas as pd
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

path = 'C:\\Users\\david\\.cache\\kagglehub\\datasets\\gzdekzlkaya\wikipedia-text-corpus-for-nlp-and-llm-projects\\versions\\1\\wikipedia_text_corpus.csv'

<>:8: SyntaxWarning: invalid escape sequence '\w'
<>:8: SyntaxWarning: invalid escape sequence '\w'
C:\Users\david\AppData\Local\Temp\ipykernel_30496\2386110537.py:8: SyntaxWarning: invalid escape sequence '\w'
  path = 'C:\\Users\\david\\.cache\\kagglehub\\datasets\\gzdekzlkaya\wikipedia-text-corpus-for-nlp-and-llm-projects\\versions\\1\\wikipedia_text_corpus.csv'


#### Método procesar

Utilizado para procesar el corpus. Se define un diccionario con los campos:
- *doc_id*: Identificador del documento procesado 
- *raw*: Texto extraido sin procesar
- *processed*: Texto procesado pero no tokenizado

Se define el stemmer y se lee el csv del corpus. Se limpia el texto y se define el data frame a partir del diccionario corpus. Por último se define una nueva fila llamada tf-idf en la que se guardarán los vectores de cada documento.

In [6]:
def procesar(path: str) -> pd.DataFrame:
    corpus = {'doc_id': [], 'raw': [], 'processed': []}
    stemmer_espanol = SnowballStemmer('english')
    df_input = pd.read_csv(path)
    for _, row in df_input.iterrows():
        text = row['text']
        clean_text = re.sub(r'[^\w\s]', '', text)
        stemmed_words = [stemmer_espanol.stem(word.lower()) for word in clean_text.split()]    
        corpus['doc_id'].append(row['Unnamed: 0'])
        corpus['raw'].append(text)
        corpus['processed'].append(' '.join(stemmed_words))

    df_final = pd.DataFrame(corpus)
    df_final['tf_idf'] = None
    return df_final

In [7]:
processed_df = procesar(path)

## Parte 1: Recuperación con TF-IDF

### Actividad:
3. Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF
4. A partir de un conjunto de 10 queries, verifica la recuperación del sistema

Se define el vectorizador y, con el método fit transform, se calcula la matriz tf-idf del corpus a partir de el texto procesado. Finalmente se guarda cada fila de la matriz (vector) en la columna correspondiente del data frame.

In [21]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(processed_df['processed'])  
processed_df['tf_idf'] = list(tfidf_matrix.toarray())

display(processed_df[['doc_id', 'processed', 'tf_idf']].head())

,doc_id,processed,tf_idf
0,1,anovo anovo former a novo is a comput servic c...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,2,batteri indic a batteri indic also known as a ...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,3,bob peas robert allen peas august 22 1940â â j...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,4,cavnet cavnet was a secur militari forum which...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,5,clidar the clidar is a scientif instrument use...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


Con la finalidad de evaluar el sistema de recuperación, se definen 10 queries, lo cuales se limpian antes de calcular la similitud. Se armó un nuevo data frame para guardar los queries y el resultado de la limpieza. Por último, se calcularon las similitudes para cada querie con cada documento

In [ ]:
queries = ['Leonardo illustrated a book on mathematical proportion in art',
           'The first one is the Toshiba DynaSheet',
           'Network of shared automated teller machines in India.',
           'This design consists of two independent LNBs in a single case.',
           'Post attained a new organization in 1906.',
           'The first project for which evidence survives was to provide a system',
           'British man was caught having sex',
           'Valmet first made tractors for Finland.',
           'The Chief Scientific Adviser to the UK government',
           'Polyaniline was first described in the mid-19th century']


clean_queries = []
stemmer_espanol = SnowballStemmer('english')
for querie in queries:
        text = querie
        clean_text = re.sub(r'[^\w\s]', '', text)
        stemmed_words = [stemmer_espanol.stem(word.lower()) for word in clean_text.split()]    
        clean_queries.append(' '.join(stemmed_words))

queries_df = pd.DataFrame({'query': queries, 'processed_query': clean_queries})
query_vectors = vectorizer.transform(queries_df['processed_query'])
similarities = cosine_similarity(query_vectors, tfidf_matrix)

Se extrajo, para cada querie, los 5 mejores documentos y se guardaron dentro de una lista, para posteriormente ubicarlos en la comluna correspondiente al data frame de los queries.

In [12]:
top_docs_list = []
for i in range(len(queries)):
    scores = similarities[i]
    best_doc_index = np.argsort(scores)[-5:][::-1]
    best_docs = processed_df.iloc[best_doc_index]['doc_id'].tolist()
    top_docs_list.append(best_docs)
queries_df['top_5_doc_ids'] = top_docs_list


In [20]:
display(queries_df[['query', 'top_5_doc_ids']].head())

,query,top_5_doc_ids
0,Leonardo illustrated a book on mathematical pr...,"[36, 516, 5476, 7053, 2902]"
1,The first one is the Toshiba DynaSheet,"[1682, 35, 4000, 7026, 7027]"
2,Network of shared automated teller machines in...,"[1928, 5892, 5150, 6690, 6288]"
3,This design consists of two independent LNBs i...,"[29, 7899, 5197, 7849, 4361]"
4,Post attained a new organization in 1906.,"[10297, 26, 6843, 10547, 6642]"


## Parte 2: Recuperación con BM25

### Actividad:
5. Implementa un sistema de recuperación usando el modelo BM25.
6. Para el mismo conjunto de 10 queries, verifica la recuperación del sistema

Para la parte de BM25 se importó la libería rank_bm25 que ofrece la facilidad del cálculo del modelo. Sin embargo, a diferencia de sklearn y su método para tf-idf, está librería exige que los documentos estén tokenizados. Es por esto que, mediante la variable tokenized_corpus, se guardadon los documentos tokenizados.

In [15]:
from rank_bm25 import BM25Okapi
tokenized_corpus = [doc.split(" ") for doc in processed_df['processed']]

Se implementó bm25 mediante BM25Okapi.

In [16]:
bm25 = BM25Okapi(tokenized_corpus)

Se hizo el mismo procedimiento para las queries, donde se tokenizó cada una de estas para posteriormente calcular los socres. Po último, se obtuvieron los mejores 5 documentos y se guardó en la columna correspondiente del data frame.

In [22]:
top_docs_list_bm25 = []

for query in queries_df['processed_query']:
    tokenized_query = query.split(" ")
    scores = bm25.get_scores(tokenized_query)
    best_doc_index = np.argsort(scores)[-5:][::-1]
    best_docs = processed_df.iloc[best_doc_index]['doc_id'].tolist()
    top_docs_list_bm25.append(best_docs)

queries_df['top_5_doc_ids_bm25'] = top_docs_list_bm25

display(queries_df[['query', 'top_5_doc_ids_bm25']].head())

,query,top_5_doc_ids_bm25
0,Leonardo illustrated a book on mathematical pr...,"[36, 3753, 1456, 6468, 343]"
1,The first one is the Toshiba DynaSheet,"[35, 1682, 4000, 6303, 5144]"
2,Network of shared automated teller machines in...,"[3156, 4136, 6599, 30, 4445]"
3,This design consists of two independent LNBs i...,"[29, 5317, 509, 6280, 9110]"
4,Post attained a new organization in 1906.,"[26, 2473, 10812, 3731, 8852]"


## Parte 3: Comparación de resultados

### Actividad:
7. Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación 

In [23]:
display(queries_df[['query', 'top_5_doc_ids', 'top_5_doc_ids_bm25']].head())

,query,top_5_doc_ids,top_5_doc_ids_bm25
0,Leonardo illustrated a book on mathematical pr...,"[36, 516, 5476, 7053, 2902]","[36, 3753, 1456, 6468, 343]"
1,The first one is the Toshiba DynaSheet,"[1682, 35, 4000, 7026, 7027]","[35, 1682, 4000, 6303, 5144]"
2,Network of shared automated teller machines in...,"[1928, 5892, 5150, 6690, 6288]","[3156, 4136, 6599, 30, 4445]"
3,This design consists of two independent LNBs i...,"[29, 7899, 5197, 7849, 4361]","[29, 5317, 509, 6280, 9110]"
4,Post attained a new organization in 1906.,"[10297, 26, 6843, 10547, 6642]","[26, 2473, 10812, 3731, 8852]"
